In [ ]:
!pip install pymupdf pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import fitz
import pandas as pd
from pathlib import Path
from tqdm import tqdm

Mounted at /content/drive


In [ ]:
pdf_dir = Path(r"/content/drive/My Drive/spj")
rows = []
for pdf_path in tqdm(list(pdf_dir.glob("*.pdf"))):
    doc = fitz.open(pdf_path)
    doc_id = pdf_path.stem  # file name without .pdf

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")

        rows.append({
            "doc_id": doc_id,
            "file_name": pdf_path.name,
            "page_no": page_num,
            "raw_text": text
        })
df = pd.DataFrame(rows)

100%|██████████| 6/6 [00:04<00:00,  1.32it/s]


In [ ]:
import re
def remove_garbage(text):
    """Remove control characters and replacement characters"""
    pattern = r'[�\uFFFD]|[\x00-\x1F\x7F-\x9F]'
    return re.sub(pattern, '', text)

def clean_whitespace(text):
    """Normalize whitespace after removing garbage"""
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["cleaned_text"] = (df["raw_text"]
                      .str.replace(r'[�\uFFFD]|[\x00-\x1F\x7F-\x9F]', '', regex=True)
                      .str.replace(r'\n+', ' ', regex=True)
                      .str.replace(r'\s+', ' ', regex=True)
                      .str.strip())

In [ ]:
df.to_csv("/content/drive/My Drive/spj.csv", index=False)

In [ ]:
df.head()

,doc_id,file_name,page_no,raw_text,cleaned_text
0,spj1,spj1.pdf,1,http://JUDIS.NIC.IN \nSUPREME COURT OF INDIA\n...,http://JUDIS.NIC.IN SUPREME COURT OF INDIAPage...
1,spj1,spj1.pdf,2,http://JUDIS.NIC.IN \nSUPREME COURT OF INDIA\n...,http://JUDIS.NIC.IN SUPREME COURT OF INDIAPage...
2,spj1,spj1.pdf,3,http://JUDIS.NIC.IN \nSUPREME COURT OF INDIA\n...,http://JUDIS.NIC.IN SUPREME COURT OF INDIAPage...
3,spj1,spj1.pdf,4,http://JUDIS.NIC.IN \nSUPREME COURT OF INDIA\n...,http://JUDIS.NIC.IN SUPREME COURT OF INDIAPage...
4,spj1,spj1.pdf,5,http://JUDIS.NIC.IN \nSUPREME COURT OF INDIA\n...,http://JUDIS.NIC.IN SUPREME COURT OF INDIAPage...


In [ ]:
df.shape

(42, 5)

In [ ]:
df["text_length"] = df["cleaned_text"].str.len()
df["text_length"].describe()

,text_length
count,42.000000
mean,2797.690476
std,815.828129
min,587.000000
25%,2522.000000
50%,2928.000000
75%,3409.000000
max,3711.000000


In [ ]:
empty_pages = (df.query("text_length < 100")
    [["doc_id", "page_no", "text_length"]]
    .head())
empty_pages

,doc_id,page_no,text_length


In [ ]:
sample = df.sample(3, random_state=42)
for _, row in sample.iterrows():
    print("=" * 80)
    print(f"Document: {row['doc_id']} | Page: {row['page_no']}")
    print(row["cleaned_text"][:1500])

Document: spj3 | Page: 10
http://JUDIS.NIC.IN SUPREME COURT OF INDIAPage 10 of 14 After the above conclusion reached in respect of thespeeches alleged to have been made by some leaders on29.1.1990 and 24.2.1990 for the reasons already given, theonly remaining findings of corrupt practice recorded by theHigh Court are based on certain wall paintings and videocassette which have been found to constitute the corruptpractices under Section 123(3) and 123(3A) of the R.P. Act.We would now examine these findings on merits. The pleading relating to the allegation of corruptpractice based on wall paintings is contained in para 21 ofthe election petition which is as under : "The petitioner states that the respondent and his agents with the consent of the respondent have also used posters, banners and wall paintings canvassing to vote for the respondent, appealing the voters to vote for the respondent in the name of Hindu religion. The petitioner has got the photographs taken of such wall paintin

In [ ]:
from collections import Counter
all_lines = [
    line.strip()
    for text in df["raw_text"]
    for line in text.split("\n")
    if len(line.strip()) > 30]
common_lines = Counter(all_lines).most_common(20)
common_lines

[('service rendered by Govt. servants.', 2),
 ('amounts  to   the  corrupt  practice  of', 2),
 ('petition  and   declaring  the   election  of  the  returned', 2),
 ('different classes  of  citizens  on  the', 2),
 ('BALAJI RAGHAVAN [IN T.C.(C) NO.9/94]S.P. ANAND [IN T.C.(C) N', 1),
 ('1996 AIR  770            1996 SCC  (1) 361', 1),
 ('JT 1995 (9)   393        1995 SCALE  (7)202', 1),
 ('1.   The short  but interesting question that arises for our', 1),
 ('"Whether the Awards, Bharat Ratna, Padma', 1),
 ('Vibhushan, Padma  Bhushan and Padma Shri', 1),
 ('(hereinafter   called    "The   National', 1),
 ('Awards") are "Titles" within the meaning', 1),
 ('of Article  18(1) of the Constitution of', 1),
 ('2.   Before dealing  with the  legal aspects of the question', 1),
 ('at issue,  we may  briefly set out the factual matrix of the', 1),
 ('two cases.  The two  petitions which have given rise to this', 1),
 ('issue were  filed in  the High  Courts of  Kerala and Madhya', 1),
 ('Pradesh

In [ ]:
summary = (
    df.groupby("doc_id", as_index=False)
      .agg(
          total_pages=("page_no", "max"),
          avg_text_length=("text_length", "mean"),
          min_text_length=("text_length", "min"),
          max_text_length=("text_length", "max")).round(2)
      .sort_values("avg_text_length", ascending=False))
summary

,doc_id,total_pages,avg_text_length,min_text_length,max_text_length
0,spj1,14,3109.36,1906,3711
2,spj3,14,2921.50,1526,3638
3,spj4,4,2731.25,1498,3637
4,spj5,5,2640.80,1276,3640
5,spj6,3,2041.67,647,3329
1,spj2,2,1408.50,587,2230


In [ ]:
!pip install -U sentence-transformers transformers faiss-cpu
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-base-en-v1.5")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
def chunk_text_sentence_aware(text, tokenizer, max_tokens=512, overlap=64):
    import re

    if not text or not text.strip():
        return []

    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_chunk = []
    current_tokens = []

    for sentence in sentences:
        sentence_tokens = tokenizer.encode(sentence, add_special_tokens=False)

        # Check if adding this sentence exceeds limit
        if len(current_tokens) + len(sentence_tokens) > max_tokens:
            # Save current chunk if it has content
            if current_chunk:
                chunk_text = ' '.join(current_chunk)
                chunks.append(chunk_text)

                # Start new chunk with overlap from previous
                overlap_sentences = current_chunk[-2:] if len(current_chunk) > 1 else current_chunk
                current_chunk = overlap_sentences + [sentence]
                current_tokens = tokenizer.encode(' '.join(current_chunk), add_special_tokens=False)
            else:
                # Single sentence is too long - fall back to token-based split
                current_chunk = [sentence]
                current_tokens = sentence_tokens
        else:
            current_chunk.append(sentence)
            current_tokens.extend(sentence_tokens)

    # Add final chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks

In [ ]:
chunk_rows = []
print("Chunking documents...")
for row in tqdm(df.itertuples(index=False), total=len(df)):
    chunks = chunk_text_sentence_aware(
        row.cleaned_text,
        tokenizer,
        max_tokens=512,
        overlap=64)

    for i, chunk in enumerate(chunks):
        chunk_rows.append({
                "doc_id": row.doc_id,
                "file_name": row.file_name,
                "page_no": row.page_no,
                "chunk_id": f"{row.doc_id}_p{row.page_no}_c{i}",
                "chunk_text": chunk})
chunk_df = pd.DataFrame(chunk_rows)
print(f"Total chunks created: {len(chunk_df)}")


Chunking documents...


100%|██████████| 42/42 [00:00<00:00, 182.47it/s]

Total chunks created: 78


In [ ]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
import faiss
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-base-en-v1.5")
model.to(device)

print("Generating embeddings...")
embeddings = model.encode(
    chunk_df["chunk_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True,
    device=device)

embeddings = np.asarray(embeddings, dtype="float32")
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)
print(f"Index created with {index.ntotal} vectors.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Index created with 78 vectors.


In [ ]:
def search_legal_docs(query, k=3):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        device=device,
        convert_to_tensor=False
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)
    results = []
    for i, idx in enumerate(indices[0]):

        if idx < len(chunk_df):
            result = chunk_df.iloc[idx]
            results.append({
                "score": float(distances[0][i]),
                "text": result["chunk_text"],
                "source": f"{result['file_name']} (Page {result['page_no']})",
                "doc_id": result["doc_id"],
                "chunk_id": result["chunk_id"]})
        return results

In [ ]:
!pip install -q bitsandbytes accelerate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
model_id = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16)

text_generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.95,
    do_sample=True,
    repetition_penalty=1.15)
print("LLM loaded successfully!")

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'repetition_penalty', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded successfully!


In [ ]:
def generate_answer(query, retrieved_results):
  if not retrieved_results:
    return "I cannot find the answer in the provided documents."
  MAX_DOCS = 5
  retrieved_results = retrieved_results[:MAX_DOCS]

    # Build context block from retrieved chunks
  context = "\n\n".join(
        f"Source: {res['source']}\nText: {res['text']}"
        for res in retrieved_results)

    # Prompt for Zephyr
  prompt = f"""<|system|>
You are an expert Indian Legal Assistant. You are given a question and a set of context documents from Indian judgments.
Answer the question strictly based on the provided context.
If the answer is not found in the context, say "I cannot find the answer in the provided documents."
Do not make up laws or facts.
Cite the Source (e.g., File Name, Page No) for your claims.

<|user|>
Context:
{context}

Question:
{query}

<|assistant|>"""

    #Generate
  outputs = text_generator(
        prompt,
        return_full_text=False)
  return outputs[0]["generated_text"].strip()

In [ ]:
def legal_chatbot(user_query,k=3, return_sources = False):
    print(f"Querying: {user_query}...")
    retrieved_docs = search_legal_docs(user_query, k=k)
    if not retrieved_docs:
        return "Sorry, no relevant documents found."

    answer = generate_answer(user_query, retrieved_docs)
    return (answer, retrieved_docs) if return_sources else answer

In [ ]:
if __name__ == "__main__":
    test_query = "What did the court say about the delay in filing the FIR?"
    answer = legal_chatbot(test_query)
    print("\n" + "=" * 50)
    print("AI Answer:\n")
    print(answer)
    print("=" * 50)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Querying: What did the court say about the delay in filing the FIR?...

AI Answer:

There is no information provided in the given context regarding the delay in filing the FIR. Hence, I cannot answer your question based on the provided context. Please provide me with the relevant context related to the delay in filing the FIR, and I will assist you accordingly.


In [ ]:
!pip install -q gradio
import gradio as gr
gr.close_all()
def predict(message, history):
    print(f"User asked: {message}")
    response = legal_chatbot(message)
    return response

demo = gr.ChatInterface(
    fn=predict,
    title="Indian Legal AI Assistant",
    description="Ask questions about the uploaded legal judgments.",
    theme="soft")
demo.launch(share=False, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User asked: in the case of BAHADURGARH PLOTHOLDERS ASSOCIATION, what percent of the balance of 
tentative price can be paid lumpsum without interest in how many days?
Querying: in the case of BAHADURGARH PLOTHOLDERS ASSOCIATION, what percent of the balance of 
tentative price can be paid lumpsum without interest in how many days?...
Keyboard interruption in main thread... closing server.


In [2]:
app_code = """
!pip install pymupdf pandas tqdm
from google.colab import drive
drive.mount('/content/drive')
import fitz
import pandas as pd
from pathlib import Path
from tqdm import tqdm

pdf_dir = Path(r"/content/drive/My Drive/spj")
rows = []
for pdf_path in tqdm(list(pdf_dir.glob("*.pdf"))):
    doc = fitz.open(pdf_path)
    doc_id = pdf_path.stem  # file name without .pdf

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")

        rows.append({
            "doc_id": doc_id,
            "file_name": pdf_path.name,
            "page_no": page_num,
            "raw_text": text
        })
df = pd.DataFrame(rows)

import re
def remove_garbage(text):
    '''Remove control characters and replacement characters'''
    pattern = r'[�\uFFFD]|[\x00-\x1F\x7F-\x9F]'
    return re.sub(pattern, '', text)

def clean_whitespace(text):
    '''Normalize whitespace after removing garbage'''
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["cleaned_text"] = (df["raw_text"]
                      .str.replace(r'[�\uFFFD]|[\x00-\x1F\x7F-\x9F]', '', regex=True)
                      .str.replace(r'\n+', ' ', regex=True)
                      .str.replace(r'\s+', ' ', regex=True)
                      .str.strip())

df.to_csv("/content/drive/My Drive/spj.csv", index=False)
df.head()
df.shape
df["text_length"] = df["cleaned_text"].str.len()
df["text_length"].describe()
empty_pages = (df.query("text_length < 100")
    [["doc_id", "page_no", "text_length"]]
    .head())
empty_pages

sample = df.sample(3, random_state=42)
for _, row in sample.iterrows():
    print("=" * 80)
    print(f"Document: {row['doc_id']} | Page: {row['page_no']}")
    print(row["cleaned_text"][:1500])

from collections import Counter
all_lines = [
    line.strip()
    for text in df["raw_text"]
    for line in text.split("\n")
    if len(line.strip()) > 30]
common_lines = Counter(all_lines).most_common(20)
common_lines

summary = (
    df.groupby("doc_id", as_index=False)
      .agg(
          total_pages=("page_no", "max"),
          avg_text_length=("text_length", "mean"),
          min_text_length=("text_length", "min"),
          max_text_length=("text_length", "max")).round(2)
      .sort_values("avg_text_length", ascending=False))
summary

!pip install -U sentence-transformers transformers faiss-cpu
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-base-en-v1.5")

def chunk_text_sentence_aware(text, tokenizer, max_tokens=512, overlap=64):
    import re

    if not text or not text.strip():
        return []

    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_chunk = []
    current_tokens = []

    for sentence in sentences:
        sentence_tokens = tokenizer.encode(sentence, add_special_tokens=False)

        # Check if adding this sentence exceeds limit
        if len(current_tokens) + len(sentence_tokens) > max_tokens:
            # Save current chunk if it has content
            if current_chunk:
                chunk_text = ' '.join(current_chunk)
                chunks.append(chunk_text)

                # Start new chunk with overlap from previous
                overlap_sentences = current_chunk[-2:] if len(current_chunk) > 1 else current_chunk
                current_chunk = overlap_sentences + [sentence]
                current_tokens = tokenizer.encode(' '.join(current_chunk), add_special_tokens=False)
            else:
                # Single sentence is too long - fall back to token-based split
                current_chunk = [sentence]
                current_tokens = sentence_tokens
        else:
            current_chunk.append(sentence)
            current_tokens.extend(sentence_tokens)

    # Add final chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks

chunk_rows = []
print("Chunking documents...")
for row in tqdm(df.itertuples(index=False), total=len(df)):
    chunks = chunk_text_sentence_aware(
        row.cleaned_text,
        tokenizer,
        max_tokens=512,
        overlap=64)

    for i, chunk in enumerate(chunks):
        chunk_rows.append({
                "doc_id": row.doc_id,
                "file_name": row.file_name,
                "page_no": row.page_no,
                "chunk_id": f"{row.doc_id}_p{row.page_no}_c{i}",
                "chunk_text": chunk})
chunk_df = pd.DataFrame(chunk_rows)
print(f"Total chunks created: {len(chunk_df)}")

from sentence_transformers import SentenceTransformer
import torch
import numpy as np
import faiss
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-base-en-v1.5")
model.to(device)

print("Generating embeddings...")
embeddings = model.encode(
    chunk_df["chunk_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True,
    device=device)

embeddings = np.asarray(embeddings, dtype="float32")
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)
print(f"Index created with {index.ntotal} vectors.")

def search_legal_docs(query, k=3):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        device=device,
        convert_to_tensor=False
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)
    results = []
    for i, idx in enumerate(indices[0]):

        if idx < len(chunk_df):
            result = chunk_df.iloc[idx]
            results.append({
                "score": float(distances[0][i]),
                "text": result["chunk_text"],
                "source": f"{result['file_name']} (Page {result['page_no']})",
                "doc_id": result["doc_id"],
                "chunk_id": result["chunk_id"]})
        return results

!pip install -q bitsandbytes accelerate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
model_id = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16)

text_generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.95,
    do_sample=True,
    repetition_penalty=1.15)
print("LLM loaded successfully!")

def generate_answer(query, retrieved_results):
  if not retrieved_results:
    return "I cannot find the answer in the provided documents."
  MAX_DOCS = 5
  retrieved_results = retrieved_results[:MAX_DOCS]

    # Build context block from retrieved chunks
  context = "\n\n".join(
        f"Source: {res['source']}\nText: {res['text']}"
        for res in retrieved_results)

    # Prompt for Zephyr
  prompt = f'''<|system|>
You are an expert Indian Legal Assistant. You are given a question and a set of context documents from Indian judgments.
Answer the question strictly based on the provided context.
If the answer is not found in the context, say "I cannot find the answer in the provided documents."
Do not make up laws or facts.
Cite the Source (e.g., File Name, Page No) for your claims.

<|user|>
Context:
{context}

Question:
{query}

<|assistant|>'''

    #Generate
  outputs = text_generator(
        prompt,
        return_full_text=False)
  return outputs[0]["generated_text"].strip()

def legal_chatbot(user_query,k=3, return_sources = False):
    print(f"Querying: {user_query}...")
    retrieved_docs = search_legal_docs(user_query, k=k)
    if not retrieved_docs:
        return "Sorry, no relevant documents found."

    answer = generate_answer(user_query, retrieved_docs)
    return (answer, retrieved_docs) if return_sources else answer

if __name__ == "__main__":
    test_query = "What did the court say about the delay in filing the FIR?"
    answer = legal_chatbot(test_query)
    print("\n" + "=" * 50)
    print("AI Answer:\n")
    print(answer)
    print("=" * 50)

!pip install -q gradio
import gradio as gr
gr.close_all()
def predict(message, history):
    print(f"User asked: {message}")
    response = legal_chatbot(message)
    return response

demo = gr.ChatInterface(
    fn=predict,
    title="Indian Legal AI Assistant",
    description="Ask questions about the uploaded legal judgments.",
    theme="soft")
demo.launch(share=False, debug=True)
"""

with open('app.py', 'w') as f:
    f.write(app_code)


<>:36: SyntaxWarning: invalid escape sequence '\s'
<>:36: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_692/3053610051.py:36: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(r'\s+', ' ', text)


In [3]:
requirements = """
torch
transformers
sentence-transformers
faiss-cpu  # or faiss-gpu if you want
gradio
pandas
numpy
bitsandbytes
accelerate
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

In [4]:
from google.colab import files
files.download('app.py')
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>